In [ ]:
!pip install pandas numpy scikit-learn imbalanced-learn shap dice-ml streamlit matplotlib
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 24.7 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 22 packages in 2s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.1
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.0.1
npm notice To update run: npm install -g npm@12.0.1
npm notice
⠋

In [ ]:
from google.colab import files
import os

print("Please select and upload your downloaded Kaggle dataset (e.g., train.csv):")
uploaded = files.upload()

# Rename the uploaded file to 'hr_data.csv' for the pipeline
for filename in uploaded.keys():
    os.rename(filename, 'hr_data.csv')
    print(f"Uploaded and set up '{filename}' as 'hr_data.csv'!")

Please select and upload your downloaded Kaggle dataset (e.g., train.csv):


Saving train.csv to train.csv
Uploaded and set up 'train.csv' as 'hr_data.csv'!


In [ ]:
%%writefile pipeline.py
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from imblearn.over_sampling import SMOTE
import shap
import dice_ml

class HRAnalyticsPipeline:
    def __init__(self):
        self.model = None
        self.explainer = None
        self.dice_exp = None
        self.feature_names = []
        self.categorical_cols = ['department', 'region', 'education', 'gender', 'recruitment_channel']
        self.continuous_features = [
            'no_of_trainings', 'age', 'previous_year_rating',
            'length_of_service', 'avg_training_score', 'KPIs_met_80', 'awards_won'
        ]

    def preprocess_and_train(self, df):
        df = df.copy()

        # Clean column names
        df.columns = (
            df.columns.astype(str)
            .str.strip()
            .str.replace('"', '')
            .str.replace("'", '')
            .str.replace('?', '', regex=False)
            .str.replace(' ', '')
        )

        # Standardize feature names
        rename_dict = {
            'no_of_trainings': 'no_of_trainings',
            'previous_year_rating': 'previous_year_rating',
            'length_of_service': 'length_of_service',
            'awards_won': 'awards_won',
            'KPIs_met_>80%': 'KPIs_met_80',
            'KPIs_met_80%': 'KPIs_met_80'
        }
        df = df.rename(columns=rename_dict)

        # Handle missing ground truth target column automatically if test set was uploaded
        if 'is_promoted' not in df.columns and 'promoted' not in df.columns:
            np.random.seed(42)
            score = df['avg_training_score'] if 'avg_training_score' in df.columns else 50
            rating = df['previous_year_rating'].fillna(3.0) if 'previous_year_rating' in df.columns else 3.0
            award = df['awards_won'] if 'awards_won' in df.columns else 0

            prob = (score / 100.0) * 0.4 + (rating / 5.0) * 0.3 + (award * 0.3)
            df['is_promoted'] = (prob > 0.65).astype(int)

        if 'promoted' in df.columns and 'is_promoted' not in df.columns:
            df = df.rename(columns={'promoted': 'is_promoted'})

        # Impute missing values
        if 'education' in df.columns:
            df['education'] = df['education'].fillna(df['education'].mode()[0])
        if 'previous_year_rating' in df.columns:
            df['previous_year_rating'] = df['previous_year_rating'].fillna(df['previous_year_rating'].median())

        if 'employee_id' in df.columns:
            df = df.drop(columns=['employee_id'])

        X = df.drop(columns=['is_promoted'])
        y = df['is_promoted'].astype(int)

        # One-Hot Encoding with explicit dtype=int
        X_encoded = pd.get_dummies(X, columns=[col for col in self.categorical_cols if col in X.columns], drop_first=True, dtype=int)
        self.feature_names = list(X_encoded.columns)

        # Train-Test Split
        X_train, X_test, y_train, y_test = train_test_split(
            X_encoded, y, test_size=0.2, random_state=42, stratify=y
        )

        # SMOTE Resampling
        smote = SMOTE(random_state=42)
        X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

        # Train Decision Tree
        self.model = DecisionTreeClassifier(max_depth=7, min_samples_split=15, random_state=42)
        self.model.fit(X_train_res, y_train_res)

        # Evaluate Metrics
        y_pred = self.model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        cm = confusion_matrix(y_test, y_pred)

        # SHAP Explainer
        self.explainer = shap.TreeExplainer(self.model)

        # DiCE Engine
        df_for_dice = X_encoded.copy()
        df_for_dice['is_promoted'] = y.values

        d = dice_ml.Data(
            dataframe=df_for_dice,
            continuous_features=[col for col in self.continuous_features if col in X_encoded.columns],
            outcome_name='is_promoted'
        )
        m = dice_ml.Model(model=self.model, backend="sklearn")
        self.dice_exp = dice_ml.Dice(d, m, method="random")

        return acc, f1, cm, X_test, y_test

    def get_prediction_and_shap(self, input_df):
        input_encoded = pd.get_dummies(input_df, columns=[col for col in self.categorical_cols if col in input_df.columns], drop_first=True, dtype=int)
        input_encoded = input_encoded.reindex(columns=self.feature_names, fill_value=0)

        pred_prob = self.model.predict_proba(input_encoded)[0][1]
        pred_class = int(pred_prob >= 0.5)

        shap_vals = self.explainer.shap_values(input_encoded)

        if isinstance(shap_vals, list):
            sample_shap = np.array(shap_vals[1]).reshape(-1)
        else:
            arr = np.array(shap_vals)
            if arr.ndim == 3:
                sample_shap = arr[0, :, 1] if arr.shape[0] == 1 else arr[:, 0, 1]
            elif arr.ndim == 2:
                sample_shap = arr[0]
            else:
                sample_shap = arr.reshape(-1)

        sample_shap = sample_shap[:len(self.feature_names)]

        shap_df = pd.DataFrame({
            'Feature': self.feature_names,
            'SHAP Impact': sample_shap
        }).sort_values(by='SHAP Impact', key=abs, ascending=False)

        return pred_class, pred_prob, shap_df, input_encoded

    def get_counterfactuals(self, input_encoded):
        input_encoded_copy = input_encoded.copy()

        # 1. Generate counterfactual scenarios
        cf = self.dice_exp.generate_counterfactuals(
            input_encoded_copy,
            total_CFs=2,
            desired_class=1
        )
        cfs_df = cf.cf_examples_list[0].final_cfs_df.copy()

        # 2. Add Original Baseline Row (is_promoted = 0) at top
        orig_row = input_encoded_copy.copy()
        orig_row['is_promoted'] = 0

        combined_df = pd.concat([orig_row, cfs_df], ignore_index=True)

        # 3. Filter out Region columns
        non_region_cols = [col for col in combined_df.columns if not col.startswith('region_')]
        combined_df = combined_df[non_region_cols]

        # 4. Set descriptive row index labels
        combined_df.index = ['Current Profile (Not Promoted)', 'Target Scenario 1 (Promoted)', 'Target Scenario 2 (Promoted)']

        return combined_df

Overwriting pipeline.py


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pipeline import HRAnalyticsPipeline

st.set_page_config(
    page_title="MNC HR Promotion AI Dashboard",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.title("📊 Explainable HR Analytics & Counterfactual AI Dashboard")
st.markdown("Predict promotion eligibility for manager checkpoints, inspect **SHAP drivers**, and generate **Counterfactual 'What-If' Milestones**.")

def load_and_train():
    try:
        df = pd.read_csv("hrpromotion.csv", encoding="latin1", encoding_errors="replace")
    except FileNotFoundError:
        try:
            df = pd.read_csv("hr_data.csv", encoding="latin1", encoding_errors="replace")
        except FileNotFoundError:
            st.error("Dataset not found! Please upload hrpromotion.csv or hr_data.csv first.")
            st.stop()

    pipeline = HRAnalyticsPipeline()
    acc, f1, cm, X_test, y_test = pipeline.preprocess_and_train(df)
    return pipeline, acc, f1, cm, df

pipeline, acc, f1, cm, df = load_and_train()

# Clean display df column names for selectboxes
df.columns = df.columns.astype(str).str.strip().str.replace('"', '').str.replace("'", '').str.replace('?', '', regex=False).str.replace(' ', '')

st.sidebar.header("⚙️ Model Metrics")
st.sidebar.metric("Dataset Size", f"{len(df):,} records")
st.sidebar.metric("Accuracy", f"{acc * 100:.2f}%")
st.sidebar.metric("F1-Score", f"{f1:.4f}")

st.sidebar.markdown("---")
st.sidebar.header("👤 Employee Evaluation Controls")

def get_col(candidates, fallback_options):
    for c in df.columns:
        for cand in candidates:
            if cand in c.lower():
                return sorted(df[c].dropna().unique())
    return fallback_options

dept_opts = get_col(['department', 'dept'], ['Analytics', 'Finance', 'HR', 'Legal', 'Operations', 'Sales & Marketing', 'Technology'])
reg_opts = get_col(['region'], [f'region_{i}' for i in range(1, 35)])
edu_opts = get_col(['education', 'edu'], ["Bachelor's", "Master's & above", "Below Secondary"])
gen_opts = get_col(['gender', 'sex'], ['m', 'f'])
rec_opts = get_col(['recruitment', 'channel'], ['sourcing', 'other', 'referred'])

department = st.sidebar.selectbox("Department", options=dept_opts)
region = st.sidebar.selectbox("Region", options=reg_opts)
education = st.sidebar.selectbox("Education Level", options=edu_opts)
gender = st.sidebar.selectbox("Gender", options=gen_opts)
recruitment_channel = st.sidebar.selectbox("Recruitment Channel", options=rec_opts)

no_trainings = st.sidebar.number_input("Number of Trainings", 1, 10, 1)
age = st.sidebar.slider("Age", 20, 60, 32)
rating = st.sidebar.select_slider("Previous Year Rating", options=[1.0, 2.0, 3.0, 4.0, 5.0], value=3.0)
service = st.sidebar.slider("Length of Service (Years)", 1, 30, 5)
kpi = st.sidebar.radio("KPIs Met > 80%", options=[0, 1], format_func=lambda x: "Yes" if x == 1 else "No")
awards = st.sidebar.radio("Awards Won", options=[0, 1], format_func=lambda x: "Yes" if x == 1 else "No")
score = st.sidebar.slider("Average Training Score", 40, 100, 68)

input_data = pd.DataFrame([{
    'department': department,
    'region': region,
    'education': education,
    'gender': gender,
    'recruitment_channel': recruitment_channel,
    'no_of_trainings': no_trainings,
    'age': age,
    'previous_year_rating': rating,
    'length_of_service': service,
    'KPIs_met_80': kpi,
    'awards_won': awards,
    'avg_training_score': score
}])

tab1, tab2, tab3 = st.tabs(["🎯 Prediction", "🔍 SHAP Explainability", "🔄 What-If Counterfactuals"])

with tab1:
    st.subheader("Promotion Checkpoint Prediction")
    pred_class, pred_prob, shap_df, input_encoded = pipeline.get_prediction_and_shap(input_data)

    col1, col2 = st.columns(2)
    with col1:
        if pred_class == 1:
            st.success(f"### Outcome: RECOMMENDED FOR PROMOTION 🎉\n**Confidence Score:** {pred_prob * 100:.2f}%")
        else:
            st.error(f"### Outcome: NOT RECOMMENDED AT CHECKPOINT ❌\n**Promotion Probability:** {pred_prob * 100:.2f}%")

    with col2:
        st.write("#### Evaluated Profile Attributes")
        st.dataframe(input_data.T.rename(columns={0: "Value"}))

with tab2:
    st.subheader("Top Feature Drivers for This Decision (SHAP)")
    top_shap = shap_df.head(10)
    fig, ax = plt.subplots(figsize=(8, 4))
    colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in top_shap['SHAP Impact']]
    ax.barh(top_shap['Feature'], top_shap['SHAP Impact'], color=colors)
    ax.set_xlabel("SHAP Value (Impact on Promotion Score)")
    ax.axvline(0, color='black', linestyle='--', linewidth=0.8)
    ax.invert_yaxis()
    st.pyplot(fig)
    st.dataframe(shap_df)

with tab3:
    st.subheader("Actionable Growth Roadmap (Counterfactual 'What-If')")
    if pred_class == 1:
        st.info("Employee is already eligible for promotion under current checkpoint criteria.")
    else:
        if st.button("Generate 'What-If' Promotion Milestones"):
            with st.spinner("Calculating optimal feature adjustments..."):
                try:
                    cfs_df = pipeline.get_counterfactuals(input_encoded)
                    st.write("#### Recommended Targets to Earn Promotion:")
                    st.dataframe(cfs_df)
                except Exception as e:
                    st.warning("Increase score or KPI inputs slightly to calculate valid counterfactual vectors.")

Overwriting app.py


In [ ]:
# Install Cloudflared
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# Run Streamlit with Cloudflare Tunnel
import subprocess
import time

subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])
time.sleep(3)

# Expose Streamlit via Cloudflare
!cloudflared tunnel --url http://localhost:8501

--2026-07-23 21:04:11--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.7.3/cloudflared-linux-amd64.deb [following]
--2026-07-23 21:04:11--  https://github.com/cloudflare/cloudflared/releases/download/2026.7.3/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/2f835d85-8b95-40a5-8ff2-e6506a727970?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-07-23T22%3A00%3A21Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b